In [0]:
# Cell 1 - Health check: Delta Lake / Unity Catalog
# Verifies connectivity to the blank Delta table created for this project.
delta_table = "workspace.foodguard_ai.delta_lake"

rows = spark.table(delta_table).count()
tbl = spark.table(delta_table)
print(f"Delta Lake connection OK - read {delta_table} ({rows} rows, blank table as expected)")
display(tbl.limit(10))

In [0]:
# Cell 2 - Health check: Lakebase Postgres (project: foodguard-ai, database: foodguard)
# Generates a short-lived OAuth token via the Databricks SDK (never printed/stored)
# and runs a SELECT 1 round trip against the Postgres endpoint.
# Note: pg8000 (pure Python) is used because importing the psycopg2-binary native
# wheel mid-session makes this serverless kernel unresponsive.
%pip install --quiet --target /tmp/pglibs pg8000

import sys
sys.path.insert(0, "/tmp/pglibs")

import pg8000.native
from databricks.sdk import WorkspaceClient

PG_HOST = "ep-plain-shape-d84tfnic-pooler.database.us-east-2.cloud.databricks.com"  # read/write pooler host
PG_DATABASE = "foodguard"
PG_USER = "yashwanth.t@ganitinc.com"  # Databricks identity; the OAuth token serves as the password

w = WorkspaceClient()
cred = w.postgres.generate_database_credential(
    endpoint="projects/foodguard-ai/branches/production/endpoints/primary"
)

conn = pg8000.native.Connection(
    user=PG_USER,
    password=cred.token,
    host=PG_HOST,
    port=5432,
    database=PG_DATABASE,
    timeout=120,
)
try:
    assert conn.run("SELECT 1")[0][0] == 1, "SELECT 1 did not return 1"
    version = conn.run("SELECT version()")[0][0]
finally:
    conn.close()

print(f"Lakebase Postgres connection OK - {PG_DATABASE} @ {PG_HOST} ({version.split(',')[0]})"